In [1]:
import asyncio
import json
import re
from pathlib import Path
from datetime import datetime
from urllib.parse import urlparse, urljoin

from playwright.async_api import async_playwright
from bs4 import BeautifulSoup

OUTPUT_DIR = Path("./output")
SCREENSHOTS_DIR = OUTPUT_DIR / "screenshots"
OUTPUT_DIR.mkdir(exist_ok=True)
SCREENSHOTS_DIR.mkdir(exist_ok=True)

BASE_URL = "https://horizn-studios.com"
CHUNK_SIZE = 500

In [2]:
async def scrape_page(url: str, label: str) -> dict:
    """
    Renders and extracts content from a single page using a headless Chromium browser.
    Captures full-page screenshot, raw HTML, and all image references.
    """
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        context = await browser.new_context(
            viewport={"width": 1440, "height": 900},
            user_agent="Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"
        )
        page = await context.new_page()
        await page.goto(url, wait_until="domcontentloaded", timeout=60000)
        await asyncio.sleep(3)

        screenshot_path = SCREENSHOTS_DIR / f"{label}.png"
        await page.screenshot(path=str(screenshot_path), full_page=True)

        html = await page.content()
        title = await page.title()

        images = []
        for img in await page.query_selector_all("img"):
            src = await img.get_attribute("src")
            alt = await img.get_attribute("alt") or ""
            if src and not src.startswith("data:"):
                images.append({
                    "url": urljoin(BASE_URL, src).split("?")[0],
                    "alt_text": alt
                })

        await browser.close()

    return {
        "url": url,
        "label": label,
        "title": title,
        "scraped_at": datetime.now().isoformat(),
        "screenshot_path": str(screenshot_path),
        "html": html,
        "images": images
    }

In [3]:
async def discover_subpage_links(url: str, link_pattern: str) -> list[dict]:
    """
    Scans a page for internal links matching a given text pattern.
    Used to dynamically identify relevant subpages rather than hardcoding URLs.
    """
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url, wait_until="domcontentloaded", timeout=60000)
        await asyncio.sleep(3)

        anchors = await page.query_selector_all("a[href]")
        discovered = []

        for anchor in anchors:
            href = await anchor.get_attribute("href")
            text = (await anchor.inner_text()).strip()

            if not href or not text:
                continue

            is_internal = href.startswith("/") or BASE_URL in href
            matches_pattern = link_pattern.lower() in text.lower() if link_pattern else True

            if is_internal and matches_pattern:
                full_url = urljoin(BASE_URL, href)
                if full_url not in [d["url"] for d in discovered]:
                    discovered.append({"url": full_url, "anchor_text": text})

        await browser.close()

    return discovered

In [4]:
def extract_text_blocks(html: str) -> list[dict]:
    """
    Extracts semantically relevant text blocks from Shopify-rendered HTML.
    Separately collects section and div containers to handle Shopify Dawn theme's
    mixed use of section and div elements for content regions.
    Footer navigation divs are explicitly excluded by heading content detection.
    Applies length filtering per semantic role – headings use a lower threshold.
    """
    soup = BeautifulSoup(html, "lxml")

    for tag in soup(["script", "style", "noscript", "svg", "iframe", "form"]):
        tag.decompose()

    section_containers = soup.find_all(
        "section",
        class_=lambda c: c is not None and "shopify-section" in c
    )
    div_containers = soup.find_all(
        "div",
        class_=lambda c: c is not None and "shopify-section" in c
    )

    footer_signals = ["shop", "about", "support", "more", "follow us", "newsletter"]

    def is_footer_section(container) -> bool:
        headings = [h.get_text(strip=True).lower() for h in container.find_all(["h2", "h3"])]
        matches = sum(1 for h in headings if any(f in h for f in footer_signals))
        return matches >= 2

    filtered_divs = [d for d in div_containers if not is_footer_section(d)]
    all_containers = section_containers + filtered_divs

    skip_signals = [
        "cookie", "provider", "shopify domain", "retention",
        "first-party", "third-party", "path /", "zendesk",
        "snowplow", "yotpo", "paypal", "cloudflare",
        "recaptcha", "domain horizn", "type first", "type third"
    ]

    blocks = []
    seen = set()

    for container in all_containers:
        for el in container.find_all(["h1", "h2", "h3", "h4", "p", "blockquote"]):
            text = re.sub(r"\s+", " ", el.get_text(separator=" ", strip=True))
            is_heading = el.name in ["h1", "h2", "h3", "h4"]
            min_length = 3 if is_heading else 25

            if len(text) < min_length or text in seen:
                continue
            if any(s in text.lower() for s in skip_signals):
                continue

            seen.add(text)
            blocks.append({
                "tag": el.name,
                "semantic_role": "heading" if is_heading else (
                    "body_text" if el.name == "p" else "quote"
                ),
                "text": text
            })

    return blocks


def extract_product_blocks(html: str) -> list[dict]:
    """
    Specialized extractor for product detail pages.
    Retains only brand-relevant content: product name, description,
    key features, and materials. Excludes shipping tables, size guides,
    and cross-sell sections. Includes customer reviews for audience analysis.
    """
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "noscript", "svg", "iframe", "form"]):
        tag.decompose()

    main = soup.find("main") or soup.find(
        "div", class_=lambda c: c and "product" in " ".join(c or [])
    )

    if not main:
        return extract_text_blocks(html)

    skip_headings = [
        "shipping", "delivery", "return", "warranty details", "size",
        "airline", "dimensions", "you'll love", "also like",
        "orders from", "express", "standard", "carbon", "over 1", "under 1"
    ]

    blocks = []
    seen = set()

    for el in main.find_all(["h1", "h2", "h3", "h4", "p", "li"]):
        text = re.sub(r"\s+", " ", el.get_text(separator=" ", strip=True))
        is_heading = el.name in ["h1", "h2", "h3", "h4"]
        min_length = 3 if is_heading else 25

        if len(text) < min_length or text in seen:
            continue
        if any(s in text.lower() for s in skip_headings):
            continue

        seen.add(text)
        blocks.append({
            "tag": el.name,
            "semantic_role": "heading" if is_heading else (
                "body_text" if el.name == "p" else "list_item"
            ),
            "text": text
        })

    return blocks

In [5]:
def create_semantic_chunks(blocks: list[dict], max_chars: int = 500) -> list[dict]:
    """
    Groups text blocks into semantically coherent chunks by heading boundaries.
    Each chunk contains one heading and its associated body content, ensuring
    the LLM receives self-contained, contextually meaningful input segments.
    """
    chunks = []
    current_heading = "Introduction"
    current_texts = []
    chunk_index = 0

    def flush(heading, texts, index):
        combined = " ".join(texts).strip()
        if not combined:
            return index
        while len(combined) > max_chars:
            split_point = combined[:max_chars].rfind(" ")
            chunks.append({
                "chunk_id": index,
                "heading": heading,
                "text": combined[:split_point].strip(),
                "char_count": split_point
            })
            combined = combined[split_point:].strip()
            index += 1
        chunks.append({
            "chunk_id": index,
            "heading": heading,
            "text": combined,
            "char_count": len(combined)
        })
        return index + 1

    for block in blocks:
        if block["semantic_role"] == "heading":
            chunk_index = flush(current_heading, current_texts, chunk_index)
            current_heading = block["text"]
            current_texts = []
        else:
            current_texts.append(block["text"])

    flush(current_heading, current_texts, chunk_index)
    return chunks


def filter_relevant_images(images: list[dict]) -> list[dict]:
    """
    Removes images that carry no brand signal and deduplicates by URL.

    Two-step filter:
        1. Skip: filenames that are definitively non-content — tracking pixels,
           video thumbnails (hash.thumbnail.*), infographics, size guides,
           and SVG icons. These are identified by reliable filename patterns,
           not by guessing content category from keywords.
        2. Deduplicate: same URL appearing on multiple pages is kept once.

    No likely_type classification is assigned. Every image that passes the
    skip filter is treated as equally valid brand content. Sampling and
    prioritisation for vision analysis are handled downstream in vision_client.py
    based on page distribution, not assumed content category.

    Returns a flat list of dicts with keys: url, alt_text.
    """
    # Filename substrings that reliably indicate non-content assets.
    # Kept minimal and based on observed corpus patterns — not speculative.
    skip_patterns = [
        # UI chrome and tracking
        "favicon", ".svg", "1x1", "pixel", "tracker", "badge", "spacer", "blank",
        # Video thumbnails generated by Shopify (hash.thumbnail.000...jpg)
        ".thumbnail.",
        # Infographics and size charts — diagrams, not photos
        "infographic", "size-guide", "size_guide",
        # Known tracking pixel present on every Horizn page
        "9156dd11",
    ]

    seen_urls = set()
    relevant = []

    for img in images:
        url_lower = img["url"].lower()
        filename = url_lower.split("/")[-1].split("?")[0]

        if any(p in filename for p in skip_patterns):
            continue
        if img["url"] in seen_urls:
            continue

        seen_urls.add(img["url"])
        relevant.append({
            "url": img["url"],
            "alt_text": img.get("alt_text", ""),
        })

    return relevant

In [6]:
async def run_pipeline():
    """
    Executes the two-stage scraping pipeline:
    Stage 1 – Manually defined entry pages (About Us, Homepage)
    Stage 2 – Subpages discovered dynamically via link pattern matching on About Us
    """
    entry_pages = [
        {"url": f"{BASE_URL}/pages/about-us", "label": "about_us"},
        {"url": BASE_URL, "label": "homepage"},
    ]

    results = {}
    for p in entry_pages:
        print(f"Scraping: {p['label']}")
        results[p["label"]] = await scrape_page(p["url"], p["label"])

    print("\nDiscovering subpages from About Us...")
    subpage_links = await discover_subpage_links(
        url=f"{BASE_URL}/pages/about-us",
        link_pattern="more about"
    )
    print(f"Found {len(subpage_links)} subpages:")
    for link in subpage_links:
        print(f"  → {link['anchor_text']} ({link['url']})")

    for link in subpage_links:
        label = re.sub(r"[^a-z0-9]+", "_", link["anchor_text"].lower()).strip("_")
        print(f"Scraping: {label}")
        results[label] = await scrape_page(link["url"], label)

    print("\nFetching representative product page...")
    product_links = await discover_subpage_links(
        url=f"{BASE_URL}/collections/luggage",
        link_pattern=""
    )
    product_links = [l for l in product_links if "/products/" in l["url"]]
    if product_links:
        first_product = product_links[0]
        label = re.sub(r"[^a-z0-9]+", "_", first_product["anchor_text"].lower()).strip("_")
        print(f"  → {first_product['url']}")
        results[label] = await scrape_page(first_product["url"], label)

    return results


def build_corpus(scraped: dict) -> dict:
    """
    Assembles the complete structured corpus from all scraped pages.
    Automatically selects the appropriate extractor based on URL pattern.
    """
    corpus = {}
    for label, data in scraped.items():
        is_product = "/products/" in data["url"]
        blocks = extract_product_blocks(data["html"]) if is_product else extract_text_blocks(data["html"])
        chunks = create_semantic_chunks(blocks, max_chars=CHUNK_SIZE)
        images = filter_relevant_images(data["images"])

        corpus[label] = {
            "meta": {
                "label": label,
                "url": data["url"],
                "title": data["title"],
                "scraped_at": data["scraped_at"],
                "screenshot_path": data["screenshot_path"],
                "page_type": "product" if is_product else "brand"
            },
            "stats": {
                "chunks": len(chunks),
                "images": len(images),
                "avg_chunk_chars": sum(c["char_count"] for c in chunks) // max(len(chunks), 1)
            },
            "chunks": chunks,
            "images": images
        }

    corpus_path = OUTPUT_DIR / "corpus.json"
    with open(corpus_path, "w", encoding="utf-8") as f:
        json.dump(corpus, f, indent=2, ensure_ascii=False)

    print(f"\nCorpus saved → {corpus_path}")
    for label, doc in corpus.items():
        print(f"  ── {label} [{doc['meta']['page_type']}]  "
              f"chunks={doc['stats']['chunks']}  images={doc['stats']['images']}")

    return corpus


# Run everything
scraped = await run_pipeline()
corpus = build_corpus(scraped)

Scraping: about_us
Scraping: homepage

Discovering subpages from About Us...
Found 3 subpages:
  → MORE ABOUT QUALITY (https://horizn-studios.com/pages/quality)
  → MORE ABOUT OUR EFFORTS AND IMPACT (https://horizn-studios.com/pages/impact)
  → MORE ABOUT OUR PARTNERS (https://horizn-studios.com/pages/partnerships)
Scraping: more_about_quality
Scraping: more_about_our_efforts_and_impact
Scraping: more_about_our_partners

Fetching representative product page...
  → https://horizn-studios.com/products/sofo-rolltop-backpack-x-night-blue-waxed-canvas

Corpus saved → output/corpus.json
  ── about_us [brand]  chunks=10  images=21
  ── homepage [brand]  chunks=13  images=48
  ── more_about_quality [brand]  chunks=9  images=23
  ── more_about_our_efforts_and_impact [brand]  chunks=5  images=19
  ── more_about_our_partners [brand]  chunks=7  images=19
  ── spotlight_sofo_backpack_rolltop_x [product]  chunks=24  images=46
